<img src="../logo.png" alt="Vegeta" width="240">

# Kestrel-X-C — the X-wing made competitive: design, CAD, CFD and FEA, against X-A, X-B and Kestrel-4Q-2

**Kestrel-X-C** keeps what the X-wing is for — a tail-sitter that stands on its tips in wind, no bracing — and spends
the rest on the cruise it lost: panels at **±30°** instead of ±45° (a panel gives back cos² φ of its lift: 0.75 instead
of 0.50), **0.6 m tapered panels**, the four roots filleted, and the **payload module** of the civilian X-wing concepts
inside one **blended fuselage** (162 × 185 mm, 0.76 m long, fineness 4.4, a flush camera window) rather than hung
under it. `X-C45` is the same airframe at the original 45°, so that the dihedral's share of the gain is visible;
`X-CX` has the panels 40 % longer in x (chord 0.21 m, the same span): what more area buys in stall speed and what the
aspect ratio costs in cruise.

| | source | method |
|---|---|---|
| layout, masses, lattice, drag, cruise, hover, stance, transitions | `kestrel_xc` on `kestrel_xwing`'s modules | analytic / lattice / Boreas / reduced-order model |
| CAD | `KestrelXC` (Dedalus) | the drawings and the CFD's STL |
| CFD: X1 `airframe-sweep`, X2 `airframe-stall` | book **kestrel-x** (`wf X1`, `wf X2`) | RANS, read here when a run is on file |
| FEA: X3 `panel` | `wf X3` | the panel spar in its socket at 4 g and its modes, read here when on file |
| Kestrel-X-A / X-B | `kestrel_xwing` recomputed | the same modules |
| Kestrel-4Q-2 | notebook 43's report | as exported |

Under the 15–30 min rule the notebook launches no solver; the chapters are optional takes of its gate (yanus_auto
produces them in screening quality when the solvers are on the machine).

In [ ]:
DEBUG = True
from vegeta import mock
DEBUG = mock.setup(DEBUG)

In [ ]:
import dataclasses, json, math, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
sys.path.insert(0, "../designs"); sys.path.insert(0, ".")
import kestrel, kestrel_systems as ks, kestrel_control as kc, kestrel_flight as kf, kestrel_transition as kt, kestrel_xwing as kx, kestrel_xc as xc, kestrel_drawings as kd
from vegeta import dedalus

pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40)
OUT = Path("../output/48_kestrel_xc"); OUT.mkdir(parents=True, exist_ok=True)
QUICK = os.environ.get("XC_QUICK") == "1"
T0 = time.time()
import yanus                                       # Ozaru's requirement checks: what this notebook needs on this machine
from yanus import Package
NEEDS = [Package("vegeta.boreas"),
         Package("vegeta.dedalus")]
print(yanus.table(yanus.check_needs(NEEDS)) if NEEDS else "needs: nothing beyond the Python environment")
BIP = xc.biplane_reference()                       # notebook 43's report (designs/data/reports/43_kestrel_study.json); None, with a loud warning, when it is missing
RUNS = dict.fromkeys(("kestrel-x/airframe-sweep@X-C", "kestrel-x/airframe-stall@X-C", "kestrel-x/panel@X-C"))   # the X1–X3 side-lab results: not read yet (kestrel-x publishes no cache key), so NOT RUN
print({k: ("on file" if v else "NOT RUN") for k, v in RUNS.items()})
MAIN = xc.MAIN

## 1. The airframe and its CAD

The layout rules (discs clear the pod and the tips, the pitch lever), the CAD's four views and the 3-D view.

In [ ]:
LAY = {v: xc.layout(v) for v in xc.VARIANTS}
L = LAY[MAIN]
keys = ("phi_deg", "panel_length", "chord", "root_chord", "tip_chord", "motor_r", "span_projected", "height", "S_ref", "AR_wing", "h", "pod_length", "pod_width", "pod_height", "prop_gap", "prop_to_pod", "prop_beyond_tip", "elevon_y_in")
display(pd.DataFrame({v: {k: Lv[k] for k in keys} | {"rules": "; ".join(xc.checks(Lv)) or "all hold"} for v, Lv in LAY.items()}).T)
display(pd.Series(xc.capsule_fit(), name="the payload module").to_frame())
display(pd.Series(xc.fuselage_geometry(), name="the blended fuselage").to_frame().T.round(4))
design = dedalus.load_design("../../designs/kestrel_xc.py:KestrelXC")
g = design.generate(**xc.cad_params(MAIN), part="aircraft")
dedalus.plot_views(g, figsize=(13, 9)); plt.suptitle(f"Kestrel-{MAIN}", y=0.99); plt.show()
fig = plt.figure(figsize=(6 * len(xc.VARIANTS), 6))
for k, v in enumerate(xc.VARIANTS, start=1):
    kd.xwing_front_view(LAY[v], ax=fig.add_subplot(1, len(xc.VARIANTS), k))
plt.tight_layout(); plt.show()
print(f"§ 1 in {time.time() - T0:.0f} s")

## 2. Masses, the CG and the panel spar

The X-wing's parts list with the module bay; each panel's spar sized as a cantilever with its tip motor at 4 g (stress,
deflection and the motor's first bending mode), the rule the FEA chapter X3 is read against.

In [ ]:
TAB = {v: xc.mass_table(Lv) for v, Lv in LAY.items()}
CI = {v: ks.cg_inertia(t) for v, t in TAB.items()}
display(pd.DataFrame({v: ks.groups(t) for v, t in TAB.items()}).round(3))
display(pd.DataFrame({v: {"take-off mass [kg]": c["mass_kg"], "CG x [mm]": c["x_cg"] * 1e3, "battery x [mm]": c["battery_x"] * 1e3, "clipped": c["battery_clipped"], "Ixx": c["Ixx"], "Iyy": c["Iyy"], "Izz": c["Izz"]} for v, c in CI.items()}).T.round(4))
SPAR = pd.DataFrame({v: xc.cantilever(Lv, mass_kg=CI[v]["mass_kg"]) for v, Lv in LAY.items()}).T
display(SPAR.round(2))
FEA = RUNS["kestrel-x/panel@X-C"]
if FEA:
    FT = pd.DataFrame(FEA["cases"]).T
    display(FT[["status", "kind", "nominal von Mises [MPa]", "peak von Mises [MPa]", "max displacement [mm]", "margin", "f1 [Hz]", "f2 [Hz]"]].rename_axis(f"X3 panel FEA [solver]: {FEA['counts']['solved']} of {FEA['counts']['n']} solved"))
    print("beam rule beside it:", {k: round(v, 2) for k, v in FEA["beam_reference"].items() if isinstance(v, float)})
else:
    print("X3 panel FEA: NOT RUN — wf X3 --variant X-C  (the beam rule above stands in; nothing here is a strength verdict)")

## 3. Aerodynamics: the lattice at 30°, the drag, the cruise — and the CFD read against it

The vertical lift slope on the panels' area against the biplane's and X-A's; the drag build-up with the filleted roots
and the blended fuselage; level flight with the thrust split that trims it. When X1 (the sweep) is on file its lift
slope, CL max and Cd0 stand beside the model, labelled; when X2 (the stall) is on file the panels' CL max is the
solver's.

In [ ]:
AER = {v: xc.aero(Lv) for v, Lv in LAY.items()}
DRG = {v: xc.drag_buildup(Lv) for v, Lv in LAY.items()}
XA = xc.xwing_rows()
BIPL = kf.aero()["biplane"]
A = pd.DataFrame({**{f"Kestrel-{v}": {"CLα vertical [1/rad]": a["CLa"], "CLα / the biplane's": a["CLa"] / BIPL["CLa"], "induced k": a["k"], "e (projected span)": a["e"], "x_np [mm]": a["x_np"] * 1e3, "Cd0": DRG[v]["Cd0"], "CdA [cm²]": DRG[v]["CdA_m2"] * 1e4} for v, a in AER.items()},
                 **{f"Kestrel-{k}": {"CLα vertical [1/rad]": r["aero"]["CLa"], "CLα / the biplane's": r["aero"]["CLa"] / BIPL["CLa"], "induced k": r["aero"]["k"], "e (projected span)": r["aero"]["e"], "x_np [mm]": r["aero"]["x_np"] * 1e3, "Cd0": r["drag"]["Cd0"], "CdA [cm²]": r["drag"]["CdA_m2"] * 1e4} for k, r in XA.items()}}).T
display(A.round(4).rename_axis("lattice + build-up"))
display(pd.DataFrame({v: {k: x * 1e4 for k, x in d["parts_m2"].items()} for v, d in DRG.items()}).T.round(1).rename(columns=lambda c: c + " [cm²]").rename_axis("drag areas [empirical build-up]"))
VEH = {v: xc.make_vehicle(v) for v in xc.VARIANTS}
CRU = {v: xc.cruise(veh) for v, veh in VEH.items()}
for v, c in CRU.items():
    print(f"Kestrel-{v}"); display(c.loc[[s for s in (10.0, 12.0, 14.0, 16.0, 20.0, 24.0) if s in c.index]].round(3))
CFD1, CFD2 = RUNS["kestrel-x/airframe-sweep@X-C"], RUNS["kestrel-x/airframe-stall@X-C"]
for name, S_ in (("X1 sweep", CFD1), ("X2 stall", CFD2)):
    if S_:
        rt = pd.DataFrame(S_["cases"]).T
        lab = "validated" if S_["counts"]["converged"] == S_["counts"]["n"] else "solver (unconverged: indicative)"
        display(rt[["alpha_deg", "status", "converged", "CL", "CD", "Cm"]].rename_axis(f"{name} [{lab}], {S_['quality']} quality"))
        if S_.get("lattice_check"):
            print(name, "against the lattice:", {k: round(v, 3) for k, v in S_["lattice_check"].items()})
    else:
        print(f"{name}: NOT RUN — wf {'X1' if 'sweep' in name else 'X2'} --variant X-C")
print(f"§ 3 in {time.time() - T0:.0f} s")

## 4. The hover and the stance

The allocation on the X-C's stations (pitch lever r sin 30°, yaw lever r cos 30°), the attainable moments, the
footprint on the four tip pods and the wind that tips it.

In [ ]:
DR = ks.drive_table()
HOV = {}
for v, Lv in LAY.items():
    ci = CI[v]
    print(f"Kestrel-{v}")
    display(kc.rank_table(kappa=float(DR.kap(0.0)), cg=ci, L=Lv)[["senses (M1..M4)", "rank", "condition", "verdict"]])
    HOV[v] = kc.hover_authority(mass_kg=ci["mass_kg"], cg=ci, inertia=ci["I"], L=Lv)
    display(HOV[v].round(3))
ST = pd.DataFrame({v: {k: x for k, x in xc.stance(Lv, mass_kg=CI[v]["mass_kg"], x_cg=CI[v]["x_cg"]).items() if k != "feet"} for v, Lv in LAY.items()}).T
display(ST.round(2).rename_axis("standing on the tip pods"))

## 5. The transitions

Notebook 40's panel model with the X-C's four panels at ±30° (the X-A/X-B rows of notebook 45 for scale): both
directions, three schedule lengths, two headwinds.

In [ ]:
durs = (1.5, 3.0) if QUICK else (1.5, 3.0, 5.0)
MAT, EPS = kt.matrix(VEH, durations=durs, headwinds=(0.0, 6.0), progress=True)
cols = ["height loss [m]", "height gain [m]", "pitch error max [deg]", "time stalled, free stream [s]", "α blown max while stalled [deg]", "moment shortfall [s]", "least pitch margin", "peak rotor thrust / T_max", "elevon max [deg]", "roll demand / authority", "pass"]
display(MAT[cols].astype({c: float for c in cols if c != "pass"}).round(2))
VER = kt.verdict(MAT)
for k, v in VER.items():
    print(f"{k}: {v['passes']} of {v['of']} pass" + ("".join(f"\n   - {w}" for w in v["failures"]) if v["failures"] else ""))
MAT.to_csv(OUT / "transition_matrix.csv")
kd.episode_figure(EPS[(f"Kestrel-{MAIN}", "forward", 3.0, 0.0)], VEH[MAIN]); plt.show()
kd.episode_figure(EPS[(f"Kestrel-{MAIN}", "back", 3.0, 0.0)], VEH[MAIN]); plt.show()
print(f"§ 5 in {time.time() - T0:.0f} s")

## 6. Against X-A, X-B and Kestrel-4Q-2

X-A and X-B recomputed from `kestrel_xwing` (the same modules, the same drive), Kestrel-4Q-2 from notebook 43's
report; then drag, power and maximum speed against airspeed for all of them.

In [ ]:
rows = {}
for v in xc.VARIANTS:
    Lv, veh, c = LAY[v], VEH[v], CRU[v]
    f = c[c["feasible"]]; c16 = c.loc[16.0] if (16.0 in c.index and c.loc[16.0]["feasible"]) else None
    hx = HOV[v]
    rows[f"Kestrel-{v}"] = {"mass [kg]": veh.mass, "wing area [m²]": Lv["S_ref"], "projected span [m]": Lv["span_projected"], "dihedral [deg]": Lv["phi_deg"], "CLα vertical": AER[v]["CLa"], "induced k": AER[v]["k"],
                            "Cd0": DRG[v]["Cd0"], "CdA [cm²]": DRG[v]["CdA_m2"] * 1e4, "stall speed [m/s]": math.sqrt(2 * veh.weight / (1.225 * Lv["S_ref"] * veh.pol.cl_max * math.cos(Lv["phi"]))),
                            "α at 16 m/s [deg]": c16["α [deg]"] if c16 is not None else np.nan, "L/D at 16 m/s": c16["L/D"] if c16 is not None else np.nan, "power at 16 m/s [W]": c16["electrical [W]"] if c16 is not None else np.nan,
                            "endurance best [min]": f["endurance [min]"].max(), "at [m/s]": f["endurance [min]"].idxmax(), "hover pitch [rad/s²]": hx.iloc[1]["angular acceleration [rad/s²]"], "hover heading [rad/s²]": hx.iloc[0]["angular acceleration [rad/s²]"],
                            "spar OD [mm]": SPAR.loc[v, "spar OD [mm]"], "tip-over [deg]": ST.loc[v, "least tip-over angle [deg]"], "wind that tips it [m/s]": ST.loc[v, "wind that tips it [m/s]"], "capsule fits": xc.capsule_fit()["capsule fits"],
                            "transitions passed": f"{VER[f'Kestrel-{v}']['passes']} of {VER[f'Kestrel-{v}']['of']}", "bracing": "none"}
XV = {k: kx.make_vehicle(next(n for n in kx.VARIANTS if n.startswith(k))) for k in XA}
for k, r in XA.items():
    veh = XV[k]; c = kx.cruise(veh); f = c[c["feasible"]]; c16 = c.loc[16.0] if (16.0 in c.index and c.loc[16.0]["feasible"]) else None
    hx = kc.hover_authority(mass_kg=veh.mass, cg={"x_cg": veh.x_cg, "y_cg": 0.0, "z_cg": veh.z_cg}, inertia=r["ci"]["I"], L=r["L"])
    st = kx.stance(r["L"], mass_kg=veh.mass, x_cg=veh.x_cg)
    rows[f"Kestrel-{k}"] = {"mass [kg]": veh.mass, "wing area [m²]": r["L"]["S_ref"], "projected span [m]": r["L"]["span_projected"], "dihedral [deg]": 45.0, "CLα vertical": r["aero"]["CLa"], "induced k": r["aero"]["k"], "Cd0": r["drag"]["Cd0"],
                            "CdA [cm²]": r["drag"]["CdA_m2"] * 1e4, "stall speed [m/s]": math.sqrt(2 * veh.weight / (1.225 * r["L"]["S_ref"] * veh.pol.cl_max * math.cos(r["L"]["phi"]))),
                            "α at 16 m/s [deg]": c16["α [deg]"] if c16 is not None else np.nan, "L/D at 16 m/s": c16["L/D"] if c16 is not None else np.nan, "power at 16 m/s [W]": c16["electrical [W]"] if c16 is not None else np.nan,
                            "endurance best [min]": f["endurance [min]"].max(), "at [m/s]": f["endurance [min]"].idxmax(), "hover pitch [rad/s²]": hx.iloc[1]["angular acceleration [rad/s²]"], "hover heading [rad/s²]": hx.iloc[0]["angular acceleration [rad/s²]"],
                            "spar OD [mm]": kx.cantilever(r["L"], mass_kg=veh.mass)["spar OD [mm]"], "tip-over [deg]": st["least tip-over angle [deg]"], "wind that tips it [m/s]": st["wind that tips it [m/s]"], "capsule fits": False,
                            "transitions passed": "see notebook 45", "bracing": "none"}
CMP = pd.DataFrame(rows).T
if BIP:
    b = BIP["variants"].get(BIP["main"], {})
    CMP.loc["Kestrel-4Q-2 (notebook 43)"] = {"mass [kg]": b.get("take-off mass [kg]"), "wing area [m²]": 0.28, "projected span [m]": 1.0, "dihedral [deg]": 0.0, "Cd0": b.get("Cd0"), "stall speed [m/s]": b.get("stall speed [m/s]"),
                                              "L/D at 16 m/s": b.get("L/D at 16 m/s"), "power at 16 m/s [W]": b.get("power at 16 m/s [W]"), "endurance best [min]": b.get("best endurance [min]"), "hover pitch [rad/s²]": BIP["hover pitch [rad/s²]"],
                                              "hover heading [rad/s²]": BIP["hover heading [rad/s²]"], "tip-over [deg]": (BIP.get("ground") or {}).get("least tip-over angle [deg]"), "capsule fits": True, "transitions passed": b.get("transitions passed"), "bracing": b.get("bracing")}
display(CMP.T)
CMP.to_csv(OUT / "comparison.csv")
P2 = kestrel.resolve(dict(elevons=True, strut_fairing=True, bracing="posts"))
ALL = {"Kestrel-4Q-2 (biplane)": kt.make_vehicle(P2, faired=True, name="4Q-2"), **{f"Kestrel-{k}": v for k, v in XV.items()}, **{f"Kestrel-{v}": veh for v, veh in VEH.items()}}
fig, axs = plt.subplots(1, 3, figsize=(17, 4.8))
DRAG = kd.drag_vs_speed(ALL, speeds=np.arange(9.0, 40.5, 0.5), axs=axs); plt.tight_layout(); plt.show()
display(DRAG.round(2))
print(f"§ 6 in {time.time() - T0:.0f} s")

## 7. Findings

Read against the tables above (recomputed on every run):

- **The dihedral is the lever.** X-C45, the same airframe at 45°, shows how much of the gain the 30° alone buys in
  lift slope, induced drag and cruise power; the fuselage, the fillets and the module account for the rest.
- **Longer in x is not better.** X-CX's 40 % more area lowers the stall speed by about a metre per second and the
  aspect ratio from 3.6 to 2.6: the induced drag rises faster than the profile drag falls, and the cruise power at
  16 m/s goes up by a fifth. Thicker panels (`t` in `drag_buildup`) cost 1–3 % drag per 3 % of thickness and buy only
  spar depth; the X-C's spar is already inside a 12 % section.
- **The module costs little once it is inside the fuselage**, and the capsule fits twice over; Kestrel-X-A could not
  take it at all. The deck stance survives: the tip pods are the feet, the tip-over angle stays above the biplane's.
- **What it does not do** is beat the biplane in cruise: cos² 30° is still 0.75, and the biplane flies its 59 W at
  16 m/s on a flat pair. Against the maritime missions of the benchmark (notebook 47) the X-C's case is the deck,
  the wind and the module together, not the transit.
- **What the solvers must settle**: the filleted roots' drag (the build-up's largest uncertain term: X1), the panels'
  CL max at 30° (X2), and the spar in its socket (X3). Until they are on file every row above is the model's.

In [ ]:
def plain(x):
    if isinstance(x, (np.floating, float)):
        return None if not np.isfinite(x) else float(x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    return x
rows_ = lambda df: {" | ".join(map(str, k)) if isinstance(k, tuple) else str(k): {kk: plain(vv) for kk, vv in r.items()} for k, r in df.T.to_dict().items()}
DESIGN = {"source": "48_kestrel_xc", "main": MAIN, "biplane_reference": BIP["source"] if BIP else None, "variants": {v: {k: x for k, x in d.items()} for v, d in xc.VARIANTS.items()},
          "layout": {v: {k: plain(x) for k, x in Lv.items() if isinstance(x, (int, float))} for v, Lv in LAY.items()}, "mass": {v: {k: plain(x) for k, x in c.items() if k != "I"} for v, c in CI.items()},
          "aero": {v: {k: plain(x) for k, x in a.items() if k != "note"} for v, a in AER.items()}, "spars": rows_(SPAR), "stance": rows_(ST), "transitions": rows_(MAT[cols]), "verdict": VER,
          "comparison": rows_(CMP.astype(object)), "drag_vs_speed": rows_(DRAG), "cfd": {k: (v["counts"] | v.get("lattice_check", {}) if v else None) for k, v in RUNS.items() if "airframe" in k},
          "fea": (FEA["cases"] if FEA else None)}
target = OUT / "kestrel_xc_design.json" if DEBUG else Path("../designs/data/kestrel_xc_design.json")
target.write_text(json.dumps(DESIGN, indent=1, default=plain))
print("design data:", target)
import kestrel_report as kr
REPORT = {"variants": pd.DataFrame(xc.VARIANTS).T, "layout": pd.DataFrame({v: {k: Lv[k] for k in keys} for v, Lv in LAY.items()}).T, "masses": pd.DataFrame({v: ks.groups(t) for v, t in TAB.items()}), "spars": SPAR,
          "aerodynamics": A, "drag areas": pd.DataFrame({v: {k: x * 1e4 for k, x in d["parts_m2"].items()} for v, d in DRG.items()}).T, "cruise tables": {v: df.loc[[s for s in (12.0, 16.0, 20.0) if s in df.index]] for v, df in CRU.items()},
          "hover authority": HOV, "stance": ST, "transitions": MAT[cols], "verdict": VER, "against X-A, X-B and Kestrel-4Q-2": CMP, "drag vs speed": DRAG,
          "CFD on file": {k: (v["counts"] if v else "NOT RUN") for k, v in RUNS.items() if "airframe" in k}, "FEA on file": (pd.DataFrame(FEA["cases"]).T if FEA else "NOT RUN"), "runtime [s]": {"total": time.time() - T0}}
EXP = kr.export("48_kestrel_xc", "Kestrel-X-C — the X-wing made competitive (against X-A, X-B and Kestrel-4Q-2)", REPORT, meta={"main": MAIN, "biplane_reference": BIP["source"] if BIP else None, "debug": DEBUG})
print(f"runtime {time.time() - T0:.0f} s")